In [135]:
import json
import pandas as pd
import os
from datetime import datetime

#코로나 관련 데이터를 가져와서 국가와 확진자수 데이터만 추출해 올것임. 그 다음에 날짜 순서대로 확진자수를 붙여가며 데이터프레임을 만들 생각.

PATH = "COVID-19-master/csse_covid_19_data/csse_covid_19_daily_reports/"  #코로나 확진자수 데이터가 들어있는 파일

Ymd = os.listdir(PATH)  #os.listdir: os 라이브러리에 있는 listdir은 PATH 경로안에 있는 csv파일들의 이름을 전부 리스트로 반환해줌. 
csv_list = list()       #이 작업을 해주는 이유는 날짜순서대로 확진자수 데이터를 추가할 것이기 때문

for d in Ymd:  #csv파일이면 csv_list에 append
    if d.split('.')[-1] == 'csv':
        csv_list.append(d)
        
csv_list.sort(key=lambda x: datetime.strptime(x, '%m-%d-%Y.csv'))  #datetime함수를 통해서 날짜순서대로 정렬함

with open('COVID-19-master/csse_covid_19_data/country_convert.json', 'r', encoding = 'utf-8-sig') as json_file:
    json_data = json.load(json_file)  #json라이브러리에 있는 load함수를 통해서 json_file에 들어있는 데이터를 딕셔너리 형태로 반환해줌
                                      #json_file 안에는 제대로된 국가명들이 적혀있음

def country_name(row):  #잘못된 국가명이면 제대로된 국가명으로 수정, 아니면 그대로 두는 함수
    if row['Country_Region'] in json_data:
        return json_data[row['Country_Region']]
    return row['Country_Region']

def dtFrame(day):  #본격적인 데이터프레임 만들기 함수, 날짜가 매개변수로 들어옴
    doc = pd.read_csv(PATH + day, encoding = 'utf-8-sig') #csv를 읽어오고, 국가와 확진자수 데이터만 가져올것임

    try:    # Country_Region이나 Country/Region으로 써져있는 경우를 발견함. 즉, 통일된 컬럼명이 아니여서 이를 통일시켜줌.
        doc = doc[['Country_Region', 'Confirmed']]
    except:
        doc = doc[['Country/Region', 'Confirmed']]
        doc.columns = ['Country_Region', 'Confirmed']
    doc = doc.dropna(subset = 'Confirmed')  #확진자수 컬럼에 누락된 값 drop
    doc['Country_Region'] = doc.apply(country_name, axis = 1)  #국가명 제대로 해주기
    doc = doc.astype({'Confirmed':'int64'})  #확진자수 데이터타입 int로 변경
    doc = doc.groupby('Country_Region').sum()  #국가를 기준으로 행을 나눔. 그리고 groupby()에는 기준으로 한 컬럼이 인덱스로 들어감.

    day = day.split('.')[0].lstrip('0').replace('-','/')  #날짜를 깔끔하게 만들어줌
    doc.columns = [day]  #Confirmed를 날짜로 변경함
    return doc

real_doc = dtFrame('01-22-2020.csv')   #첫 번째 날짜의 데이터를 먼저 변수에 넣어줌

for i in csv_list:
    if i == '01-22-2020.csv':   #첫 번째 날짜의 데이터가 i에 들어가면 패스하고 이어서 계속 for문 진행
        continue
    a = dtFrame(i)
    real_doc = pd.merge(real_doc, a, how='outer', left_index = True, right_index = True) #outer 방식, 인덱스를 기준으로 데이터프레임을 합쳐준다.
real_doc = real_doc.fillna(0)  #결측치 값을 0으로 채워줌.

In [136]:
real_doc

,1/22/2020,1/23/2020,1/24/2020,1/25/2020,1/26/2020,1/27/2020,1/28/2020,1/29/2020,1/30/2020,1/31/2020,...,2/28/2023,3/01/2023,3/02/2023,3/03/2023,3/04/2023,3/05/2023,3/06/2023,3/07/2023,3/08/2023,3/09/2023
Country_Region,,,,,,,,,,,,,,,,,,,,,
Afghanistan,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,209322.0,209340.0,209358.0,209362.0,209369.0,209390.0,209406.0,209436.0,209451.0,209451.0
Albania,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,334391.0,334408.0,334408.0,334427.0,334427.0,334427.0,334427.0,334427.0,334443.0,334457.0
Algeria,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,271441.0,271448.0,271463.0,271469.0,271469.0,271477.0,271477.0,271490.0,271494.0,271496.0
Andorra,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,47866.0,47875.0,47875.0,47875.0,47875.0,47875.0,47875.0,47875.0,47890.0,47890.0
Angola,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,105255.0,105277.0,105277.0,105277.0,105277.0,105277.0,105277.0,105277.0,105288.0,105288.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
West Bank and Gaza,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,703228.0,703228.0,703228.0,703228.0,703228.0,703228.0,703228.0,703228.0,703228.0,703228.0
Winter Olympics 2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,535.0,535.0,535.0,535.0,535.0,535.0,535.0,535.0,535.0,535.0
Yemen,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,11945.0,11945.0,11945.0,11945.0,11945.0,11945.0,11945.0,11945.0,11945.0,11945.0


In [137]:
#이번엔 국기를 담고있는 링크를 위에 만들어둔 데이터프레임에 붙여줄거임.

country_info = pd.read_csv("COVID-19-master/csse_covid_19_data/UID_ISO_FIPS_LookUp_Table.csv", encoding='utf-8-sig', keep_default_na=False, na_values='')
country_info.head()

country_info = country_info[['iso2','Country_Region']]
c = country_info.drop_duplicates(subset = 'Country_Region', keep = 'last')   #'Country_Region'열을 기준으로 중복된 값이 있으면 마지막 행만 남겨두고 나머지는 제거
c = c.dropna(subset = ['iso2'])

def create_flag_link(column):   #여기서 만든 링크가 국기를 담고있는 링크임
    flag_link = 'https://public.flourish.studio/country-flags/svg/' + column.lower() + '.svg'
    print(flag_link)
    return flag_link

c['iso2'] = c['iso2'].apply(create_flag_link)   #iso2 열에 그 링크를 넣어줌. 이러면 각 행마다 들어있는 데이터에 함수가 적용됨.
c = c.set_index('Country_Region')
real_final = pd.merge(real_doc, c, how='left', left_index = True, right_index = True)  #위의 데이터프레임과 합쳐주기 이번엔 데이터프레임(left)을 기준으로 합침
real_final = real_final.dropna(subset = 'iso2')  #링크가 배치가 안된 행은 drop

https://public.flourish.studio/country-flags/svg/bw.svg
https://public.flourish.studio/country-flags/svg/bi.svg
https://public.flourish.studio/country-flags/svg/sl.svg
https://public.flourish.studio/country-flags/svg/af.svg
https://public.flourish.studio/country-flags/svg/al.svg
https://public.flourish.studio/country-flags/svg/dz.svg
https://public.flourish.studio/country-flags/svg/ad.svg
https://public.flourish.studio/country-flags/svg/ao.svg
https://public.flourish.studio/country-flags/svg/ag.svg
https://public.flourish.studio/country-flags/svg/ar.svg
https://public.flourish.studio/country-flags/svg/am.svg
https://public.flourish.studio/country-flags/svg/at.svg
https://public.flourish.studio/country-flags/svg/az.svg
https://public.flourish.studio/country-flags/svg/bs.svg
https://public.flourish.studio/country-flags/svg/bh.svg
https://public.flourish.studio/country-flags/svg/bd.svg
https://public.flourish.studio/country-flags/svg/bb.svg
https://public.flourish.studio/country-flags/svg

In [146]:
#마지막으로 컬럼들의 위치를 변경해줌. 링크가 맨 앞에 왔으면 좋겠음.
c_list = real_final.columns
b = c_list[:1143]
b = b.insert(0, 'iso2')

Index(['iso2', 'iso2', '1/22/2020', '1/23/2020', '1/24/2020', '1/25/2020',
       '1/26/2020', '1/27/2020', '1/28/2020', '1/29/2020',
       ...
       '2/27/2023', '2/28/2023', '3/01/2023', '3/02/2023', '3/03/2023',
       '3/04/2023', '3/05/2023', '3/06/2023', '3/07/2023', '3/08/2023'],
      dtype='object', length=1144)

In [139]:
real_final = real_final[b]   #위치 변경

In [140]:
real_final.to_csv('COVID-19-master/i_do_final.csv',encoding = 'utf-8-sig') #to_csv('파일명', 인코딩방식)을 통해서 csv로 저장